In [ ]:

# === Fast, low-VRAM streaming with TorchCodec VideoDecoder (no NumPy) + text prompts ===
# Requirements:
#   pip install --pre transformers accelerate
#   pip install torchcodec            # TorchCodec uses your system FFmpeg/NVDEC
# Make sure you have access to the "facebook/sam3" checkpoints.

import torch
from accelerate import Accelerator

from transformers import Sam3VideoModel, Sam3VideoProcessor
from torchcodec.decoders import VideoDecoder  # TorchCodec frame iterator
from tqdm import tqdm 

# ---------------------------
# Helper: iterate frames from TorchCodec, staying in torch (no NumPy)
# ---------------------------
def iter_video_frames_torchcodec(
    path: str,
    *,
    decode_device: str = "cpu",      # "cpu" or "cuda" (CUDA uses NVDEC if available)
    stride: int = 1,
    max_frames: int | None = None,
    to_hwc: bool = True,             # SAM-3 processor happily accepts HWC torch.uint8
):
    """
    Yields (frame_index, frame_tensor) where frame_tensor is torch.uint8.
    If to_hwc=True → [H, W, 3] RGB; else → [3, H, W] RGB.
    """
    dec = VideoDecoder(path, device=decode_device)   # TorchCodec → FFmpeg/NVDEC backend
    total = dec.metadata.num_frames                  # random access; no preloading
    emitted = 0
    if not max_frames:
        max_frames == total
    for i in tqdm(range(0, total, stride)):
        # frame_chw is torch.uint8 [C,H,W] on decode_device (cpu or cuda)
        frame_chw = dec[i]
        if to_hwc:
            frame = frame_chw.permute(1, 2, 0).contiguous()   # torch uint8 HWC
        else:
            frame = frame_chw
        yield i, frame

        emitted += 1
        if max_frames is not None and emitted >= max_frames:
            break

# ---------------------------
# Model + processor (Transformers SAM-3 Video)
# ---------------------------
accel  = Accelerator()
device = accel.device                    # e.g., cuda:0
dtype  = torch.bfloat16                  # good trade-off for VRAM

model = Sam3VideoModel.from_pretrained("facebook/sam3").to(device, dtype=dtype)
processor = Sam3VideoProcessor.from_pretrained("facebook/sam3")

# ---------------------------
# Start a *streaming* session (no video passed → no preload)
# Keep temporal state on CPU to minimize VRAM.
# ---------------------------
inference_session = processor.init_video_session(
    inference_device=device,
    inference_state_device="cpu",        # tracker state on CPU
    processing_device="cpu",
    video_storage_device="cpu",
    dtype=dtype,
)

# ---------------------------
# Text prompts (PCS): one concept per call
# ---------------------------
for phrase in ["chicken", "bird"]:
    inference_session = processor.add_text_prompt(
        inference_session=inference_session,
        text=phrase,
    )

# ---------------------------
# Stream frames directly as torch tensors (no NumPy),
# optionally decoding on GPU to avoid H→D copies.
# ---------------------------
VIDEO = "../data/test.mp4"
MAX_FRAMES = 10
DECODE_ON = "cuda"       # "cuda" (NVDEC when available) or "cpu"

processed_outputs = {}

for fidx, frame in iter_video_frames_torchcodec(
    VIDEO, decode_device=DECODE_ON, stride=1, to_hwc=True
):
    # If you decoded on CUDA and your model is on the same device,
    # you can pass the CUDA tensor directly: the processor will keep it on 'device'.
    # frame: torch.uint8 [H,W,3] RGB on decode_device
    # Build processor inputs (keeps everything in torch)
    inputs = processor(images=frame, device=device, return_tensors="pt")

    # Streaming step: push this single frame to the model
    sam3_out = model(
        inference_session=inference_session,
        frame=inputs.pixel_values[0],    # already on 'device'
        reverse=False,
    )

    # Post-process back to original spatial size
    outputs = processor.postprocess_outputs(
        inference_session,
        sam3_out,
        original_sizes=inputs.original_sizes,
    )

    processed_outputs[fidx] = outputs

    # masks = outputs.get("masks", None)       # [N, H, W] at original resolution
    # ids   = outputs.get("object_ids", None)  # [N]
    # if masks is not None:
    #     print(f"Frame {fidx}: masks {tuple(masks.shape)}, ids {tuple(ids.shape) if ids is not None else '—'}")
    # else:
    #     print(f"Frame {fidx}: no masks (keys={list(outputs.keys())})")


Loading weights:   0%|          | 0/1797 [00:00<?, ?it/s]

  2%|▏         | 15/733 [00:02<01:49,  6.53it/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 733/733 [03:31<00:00,  3.46it/s]


: 

In [ ]:
processed_outputs[0]